# Module 1: Python Fundamentals for Infrastructure (The Basics)

**Concepts in this module:** Variables · Strings & integers · Lists · Dictionaries (JSON-like) · Tuples · Sets · `if`/`else` · `for` / `while` loops

**Audience:** DevOps, Platform Engineering, and MLOps — you know kubectl, CI, and YAML; this module makes Python syntax feel like a tool, not homework.

---

### How this notebook is structured

| Cell type | What you do |
|-----------|-------------|
| **Markdown** | Theory + where you use it in infra/ML platforms. Read first. |
| **Code** | Short, runnable examples. **Edit values and re-run** (`Shift+Enter`). |

**Learning rhythm:** a few paragraphs → run one small cell → next topic. No huge code dumps without explanation.

**Red thread:** messy text in → lists/dicts → `for` loop → `if/else` decision (rollout, skip, alert, promote model). Same pattern in Modules 2+ with functions and APIs.

## 1. Variables and core types

A **variable** is a name for a value (`cluster = "ncs57"`), like bash `VAR=value`, except Python remembers the **type** of the value.

**Why you care:** scripts glue together cluster names, ports, feature flags, model IDs, and booleans (prod? dry-run? pod ready?). One name → fewer copy-paste bugs in CI and runbooks.

| Type | Example | DevOps / Platform / MLOps |
|------|---------|----------------------------|
| `str` | `"10.8.116.50"`, `"v1.2.3"` | hostnames, image tags, namespaces, log text |
| `int` | `22`, `6443`, `200` | ports, replicas, HTTP status, exit codes |
| `float` | `1.45`, `0.001` | load, latency, `learning_rate`, metric values |
| `bool` | `True`, `False` | maintenance mode, `use_gpu`, Ready condition |

**Style:** `snake_case` for variables. **`UPPER_SNAKE`** for constants (`MAX_RETRIES = 5`) — team convention, not enforced by Python.

**Debug:** `type(x)` when CI prints something unexpected.

In [4]:
# Platform example — change values and re-run
cluster_name = "ncs57"
api_server_ip = "10.8.116.50"
ssh_port = 22
k8s_api_port = 6443
maintenance_mode = False
cpu_load_1m = 1.45

print(type(cluster_name), type(k8s_api_port), type(maintenance_mode))
print(f"{cluster_name} API https://{api_server_ip}:{k8s_api_port} maint={maintenance_mode}")

<class 'str'> <class 'int'> <class 'bool'>
ncs57 API https://10.8.116.50:6443 maint=False


**MLOps:** same types in job config — model name (`str`), batch size (`int`), learning rate (`float`), GPU flag (`bool`). Later modules load these from files/APIs; here, get used to **naming and printing**.

In [2]:
model_name = "new QA guys"
batch_size = 32
learning_rate = 0.001
use_gpu = True
print(f"train {model_name} batch={batch_size} lr={learning_rate} gpu={use_gpu}")

train new QA guys batch=32 lr=0.001 gpu=True


## 2. Strings — parse script and API output

Most day-one Python in infra is **string wrangling**: split host lists, trim `kubectl` output, build URLs for health checks.

Strings are **immutable** — `s.strip()` returns a new string; use `s = s.strip()` to keep it.

| Tool | Typical use |
|------|-------------|
| `.strip()` | remove trailing newline/spaces |
| `.split(",")` | CSV, comma-separated IPs |
| f-strings | `f"GET http://{host}:{port}/health"` |
| `{x!r}` | debug hidden spaces: `f"{x!r}"` |

**Footgun:** `split(",")` leaves spaces — **strip each part** when input is human or legacy bash.

In [6]:
raw_ips = " 10.0.0.1, 192.168.1.5 , 172.16.0.2  "
print("raw:", raw_ips)
print(f"raw_no_spaces: {raw_ips!r}")

ip_list = raw_ips.split(",")
print("after split:", ip_list)

print("second IP cleaned:", ip_list[1].strip())

raw:  10.0.0.1, 192.168.1.5 , 172.16.0.2  
raw_no_spaces: ' 10.0.0.1, 192.168.1.5 , 172.16.0.2  '
after split: [' 10.0.0.1', ' 192.168.1.5 ', ' 172.16.0.2  ']
second IP cleaned: 192.168.1.5


**List comprehension** — compact “clean every item” pattern you will see in real repos:

`[ip.strip() for ip in raw.split(",") if ip.strip()]` → read as: *for each part, strip; keep non-empty*.

In [7]:
ssh_port = 22
clean_ips = [ip.strip() for ip in raw_ips.split(",") if ip.strip()]
print("clean list:", clean_ips)
print(f"ssh {clean_ips[0]} {ssh_port}")

clean list: ['10.0.0.1', '192.168.1.5', '172.16.0.2']
ssh 10.0.0.1 22


**Slice / split trick:** `image.split(":")[-1]` → tag; `namespace[0]` → first character. Prefer **split** on `.` or `/` for structured strings.

In [8]:
namespace = "iim-lhp"
image = "registry.example.com/ml/train:v1.2.3"
print("namespace starts with:", namespace[0])
print("tag:", image.split(":")[-1])

namespace starts with: i
tag: v1.2.3


## 3. Lists — ordered targets and stages

Lists keep **order** and are **mutable** (`append`, sort, replace).

- **Platform:** rollout order, ports to scan, pod names from one line of output.
- **MLOps:** pipeline stage names, experiment IDs, paths before pandas/paths libs.

| Op | Meaning |
|----|--------|
| `xs[0]`, `xs[-1]` | first / last |
| `xs.append(x)` | add end |
| `len(xs)` | count |
| `for x in xs:` | process in order |
| `enumerate(xs, start=1)` | index + value for logs |

In [9]:
workers = ["worker-01", "worker-02", "worker-03"]
workers.append("worker-04")
for w in workers:
    print("drain?", w)

drain? worker-01
drain? worker-02
drain? worker-03
drain? worker-04


In [13]:
stages = ["validate_data", "train", "evaluate", "register_model", "finish", "start_again"]
for n, stage in enumerate(stages, start=1):
    print(f"[{n}/{len(stages)}] {stage}")

[1/6] validate_data
[2/6] train
[3/6] evaluate
[4/6] register_model
[5/6] finish
[6/6] start_again


## 4. Dictionaries — same shape as JSON / K8s / ML metadata

Dict = **key → value**. Nested dicts + lists match API JSON, `kubectl -o json`, pipeline responses, MLflow run payloads.

- `d["key"]` — must exist or **KeyError**
- `d.get("key", default)` — optional fields
- `for k, v in d.items():` — iterate pairs

JSON uses `null`/`true`/`false`; Python uses `None`/`True`/`False` (Module 6+ often uses `import json`).

In [16]:
node = {
    "hostname": "bcmt-ncs57",
    "ip": "10.8.116.50",
    "roles": ["control-plane", "worker"],
    "ssh_port": 22,
    "maintenance": False,
    "gpu": "NVidia"
}
print(node["ip"])
print("gpu:", node.get("gpu", "none"))
for role in node["roles"]:
    print(node["hostname"], role)

10.8.116.50
gpu: NVidia
bcmt-ncs57 control-plane
bcmt-ncs57 worker


In [17]:
# This dictionary represents one ML experiment/run.
# It is shaped like JSON: keys point to values, and some values are nested dictionaries.
run = {
    "run_id": "a1b2c3",
    "experiment": "fraud-detection",
    "metrics": {"auc": 0.91, "latency_p99_ms": 42.5},
    "tags": {"env": "staging", "team": "ml-platform"},
}

# .items() gives each key/value pair from the tags dictionary.
# The loop prints every tag in a readable key=value format.
for k, v in run["tags"].items():
    print(f"tag {k}={v}")

# Access a nested dictionary value:
# first get "metrics", then get the "auc" value inside it.
print("auc", run["metrics"]["auc"])


tag env=staging
tag team=ml-platform
auc 0.91


## 5. Tuples — fixed `(host, port)` records

Immutable ordered bundle. Unpacking: `host, port = endpoint`.

Use when the pair should not accidentally grow (contrast with lists for inventory).

In [18]:
api = ("10.8.116.50", 6443)
host, port = api
print(f"https://{host}:{port}")
version = (1, 28, 4)
print("version:", ".".join(str(n) for n in version))

https://10.8.116.50:6443
version: 1.28.4


## 6. Sets — unique hosts and label diffs

Unordered **unique** values. Dedupe alerts; `required - present` for missing labels; `a & b` for overlap.

`x in my_set` is the fast membership check.

In [19]:
hosts = ["node-a", "node-b", "node-a", "node-c"]
print(set(hosts))
required = {"env", "app", "team"}
on_pod = {"env", "app", "version"}
print("missing:", required - on_pod)

{'node-b', 'node-c', 'node-a'}
missing: {'team'}


## 7. `if` / `elif` / `else` — policy in code

Indent **4 spaces**. Combine with `and` / `or`. Falsy values include empty string, `[]`, `0`, and `None`.

- **Platform:** prod + high load → defer change.
- **MLOps:** metric gate before promote.

In [20]:
def action(env: str, load: float) -> str:
    if env == "prod" and load > 1.0:
        return "defer (prod hot)"
    if load > 2.0:
        return "page"
    if load > 1.0:
        return "watch"
    return "proceed"

for env, load in [("prod", 1.2), ("lab", 0.4), ("lab", 2.5)]:
    print(env, load, "->", action(env, load))

prod 1.2 -> defer (prod hot)
lab 0.4 -> proceed
lab 2.5 -> page


In [21]:
auc, min_auc = 0.88, 0.85
drift = False
if drift:
    print("block: drift")
elif auc >= min_auc:
    print("promote staging")
else:
    print("stay in dev")

promote staging


## 8. `for` loops — every pod, every code, every row

`for item in sequence:` is the default. `range(1, n+1)` for attempt counters. Build lists with `.append()` inside the loop when filtering.

In [23]:
pods = [
    {"name": "api-7fd9", "ready": False},
    {"name": "api-7fe0", "ready": False},
    {"name": "cache-0", "ready": True},
]
bad = []
for p in pods:
    if not p["ready"]:
        bad.append(p["name"])
print("not ready:", bad)

not ready: ['api-7fd9', 'api-7fe0']


In [24]:
for attempt in range(1, 4):
    print(f"probe attempt {attempt}/3")

probe attempt 1/3
probe attempt 2/3
probe attempt 3/3


## 9. `while` — poll until ready

When count is unknown: wait for deployment, job status, worker drain.

`break` = stop early on success; `continue` = skip to next iteration. **Must** change something each loop or you hang forever.

In [34]:
left, attempt, up = 5, 0, False
while left > 0 and not up:
    attempt += 1
    left -= 1
    print(f"poll {attempt}")
    if attempt >= 3:
        up = True
print("ok" if up else "timeout")

retries, status = 10, False
while not status:
    print("kubectl get pods")
    retries -= 1
    if retries == 0:
        status = False
        break
print("pod is up" if status else "timeout")


poll 1
poll 2
poll 3
ok
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
kubectl get pods
timeout


## 10. Capstone — parse → dicts → decide

Try editing `inventory_csv`: add a service or extra spaces. Predict output before you run.

In [35]:
inventory_csv = "web:10.0.0.5:80, api:10.0.0.6:8080 , db:10.0.0.7:5432"
services = []
for chunk in inventory_csv.split(","):
    chunk = chunk.strip()
    if not chunk:
        continue
    name, ip, port_s = chunk.split(":")
    services.append({"name": name.strip(), "ip": ip.strip(), "port": int(port_s.strip())})

for svc in services:
    if svc["port"] == 5432:
        act = "backup first"
    elif svc["port"] < 1024:
        act = "check firewall"
    else:
        act = "normal deploy"
    print(f"{svc['name']} {svc['ip']}:{svc['port']} -> {act}")

web 10.0.0.5:80 -> check firewall
api 10.0.0.6:8080 -> normal deploy
db 10.0.0.7:5432 -> backup first


---
# Exercises

Markdown = task. Next code cell = **your** solution (`# TODO`). Solutions at the bottom — peek only after trying.

**Ex 1** — Split `raw` on `;`, strip, skip empty; unique IPs in a `set`; print count + sorted list.

`raw = " 192.168.0.1;192.168.0.2 ; 192.168.0.1 ;10.0.0.1 "`

In [ ]:
raw = " 192.168.0.1;192.168.0.2 ; 192.168.0.1 ;10.0.0.1 "
# TODO


**Ex 2** — Dict `node` (`name`, `ip`, `env`, `cpu_load`). One printed line:  
`SKIP prod maintenance window` if prod and load>1.0; else `SCALE cautiously` if load>1.5; else `OK`.

In [ ]:
# TODO


**Ex 3** — Count ok (200–299), client_error (400–499), server_error (500–599) in `statuses`; ignore others.

In [ ]:
statuses = [200, 200, 503, 404, 200, 500]
# TODO


**Ex 4** — `while retries < 4`: print `attempt N`, break on N==2.

In [ ]:
# TODO


**Ex 5** — Print `name`, `version` for `clusters` where `region=="eu"` and `version>=1.27`.

In [ ]:
clusters = [
    {"name": "ncs57", "region": "eu", "version": 1.28},
    {"name": "ncs1", "region": "eu", "version": 1.26},
    {"name": "ncs10", "region": "us", "version": 1.29},
]
# TODO


**Ex 6** — For `(host, port)` in `endpoints`, print `connect host:port`.

In [ ]:
endpoints = [("10.0.0.1", 22), ("10.0.0.2", 6443)]
# TODO


---
# Solutions

In [ ]:
raw = " 192.168.0.1;192.168.0.2 ; 192.168.0.1 ;10.0.0.1 "
u = set()
for p in raw.split(";"):
    ip = p.strip()
    if ip:
        u.add(ip)
s = sorted(u)
print(len(s), s)

In [ ]:
node = {"name": "w3", "ip": "10.0.0.12", "env": "prod", "cpu_load": 1.2}
if node["env"] == "prod" and node["cpu_load"] > 1.0:
    print("SKIP prod maintenance window")
elif node["cpu_load"] > 1.5:
    print("SCALE cautiously")
else:
    print("OK")

In [ ]:
statuses = [200, 200, 503, 404, 200, 500]
ok = ce = se = 0
for c in statuses:
    if 200 <= c <= 299: ok += 1
    elif 400 <= c <= 499: ce += 1
    elif 500 <= c <= 599: se += 1
print(ok, ce, se)

In [ ]:
r = 0
while r < 4:
    r += 1
    print(f"attempt {r}")
    if r == 2:
        break

In [ ]:
clusters = [
    {"name": "ncs57", "region": "eu", "version": 1.28},
    {"name": "ncs1", "region": "eu", "version": 1.26},
    {"name": "ncs10", "region": "us", "version": 1.29},
]
for c in clusters:
    if c["region"] == "eu" and c["version"] >= 1.27:
        print(c["name"], c["version"])

In [ ]:
endpoints = [("10.0.0.1", 22), ("10.0.0.2", 6443)]
for host, port in endpoints:
    print(f"connect {host}:{port}")

**Next steps:** `Kernel → Restart & Run All`. Stuck? Print with `!r`, check `type()`, shrink the cell to three lines. Module 2 adds functions so you don’t repeat the same `if` blocks.